In [1]:
pip install nba_api pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd

gamelog = leaguegamelog.LeagueGameLog(season='2023-24', season_type_all_star='Regular Season')
df = gamelog.get_data_frames()[0]

print(df.head())

  SEASON_ID     TEAM_ID TEAM_ABBREVIATION              TEAM_NAME     GAME_ID  \
0     22023  1610612743               DEN         Denver Nuggets  0022300061   
1     22023  1610612744               GSW  Golden State Warriors  0022300062   
2     22023  1610612747               LAL     Los Angeles Lakers  0022300061   
3     22023  1610612756               PHX           Phoenix Suns  0022300062   
4     22023  1610612740               NOP   New Orleans Pelicans  0022300071   

    GAME_DATE      MATCHUP WL  MIN  FGM  ...  DREB  REB  AST  STL  BLK  TOV  \
0  2023-10-24  DEN vs. LAL  W  240   48  ...    33   42   29    9    6   12   
1  2023-10-24  GSW vs. PHX  L  240   36  ...    31   49   19   11    6   11   
2  2023-10-24    LAL @ DEN  L  240   41  ...    31   44   23    5    4   12   
3  2023-10-24    PHX @ GSW  W  240   42  ...    43   60   23    5    7   19   
4  2023-10-25    NOP @ MEM  W  240   40  ...    41   52   22    8    5   21   

   PF  PTS  PLUS_MINUS  VIDEO_AVAILABLE  
0 

In [9]:
from time import sleep

seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2015, 2025)]

all_games = []

for season in seasons:
    print(f"Fetching {season}...")
    
    gamelog = leaguegamelog.LeagueGameLog(
        season=season,
        season_type_all_star='Regular Season'
    )
    
    df = gamelog.get_data_frames()[0]
    df['SEASON'] = season
    
    all_games.append(df)
    
    sleep(1)  # avoid rate limiting

games = pd.concat(all_games, ignore_index=True)

Fetching 2015-16...
Fetching 2016-17...
Fetching 2017-18...
Fetching 2018-19...
Fetching 2019-20...
Fetching 2020-21...
Fetching 2021-22...
Fetching 2022-23...
Fetching 2023-24...
Fetching 2024-25...


In [10]:
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])

# Keep only useful columns (you can expand later)
games = games[[
    'GAME_ID',
    'GAME_DATE',
    'TEAM_ID',
    'TEAM_ABBREVIATION',
    'PTS',
    'MATCHUP',
    'SEASON'
]]

In [11]:
games['IS_HOME'] = games['MATCHUP'].str.contains('vs.')

In [15]:
home = games[games['IS_HOME']].copy()
away = games[~games['IS_HOME']].copy()

df_games = home.merge(
    away,
    on='GAME_ID',
    suffixes=('_HOME', '_AWAY')
)

In [17]:
df_games = df_games[[
    'GAME_ID',
    'GAME_DATE_HOME',
    'TEAM_ABBREVIATION_HOME',
    'TEAM_ABBREVIATION_AWAY',
    'PTS_HOME',
    'PTS_AWAY'
]].rename(columns={'GAME_DATE_HOME': 'GAME_DATE'})

In [19]:
df_games.to_csv('nba_games.csv', index=False)

In [21]:
print(df_games.head())
print(df_games.shape)

      GAME_ID  GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
0  0021500002 2015-10-27                    CHI                    CLE   
1  0021500003 2015-10-27                    GSW                    NOP   
2  0021500001 2015-10-27                    ATL                    DET   
3  0021500005 2015-10-28                    BOS                    PHI   
4  0021500010 2015-10-28                    HOU                    DEN   

   PTS_HOME  PTS_AWAY  
0        97        95  
1       111        95  
2        94       106  
3       112        95  
4        85       105  
(11974, 6)


In [3]:
from tqdm import tqdm
from time import sleep
from nba_api.stats.endpoints import boxscoretraditionalv3

import pandas as pd
import os
import time
from nba_api.stats.endpoints import boxscoretraditionalv3
from tqdm import tqdm

OUTPUT_FILE = "boxscores_stream.csv"
FAILED_FILE = "failed_games.csv"

game_ids = df_games["GAME_ID"].unique()

# -----------------------
# LOAD PROGRESS (RESUME)
# -----------------------
if os.path.exists(OUTPUT_FILE):
    done = pd.read_csv(OUTPUT_FILE)["GAME_ID"].unique()
    done = set(done)
else:
    done = set()

failed_games = []

# -----------------------
# SCRAPE LOOP
# -----------------------
for game_id in tqdm(game_ids, desc="Streaming NBA data"):

    if game_id in done:
        continue  # skip already saved games

    try:
        box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id)
        df = box.get_data_frames()[0]
        df["GAME_ID"] = game_id

        # -----------------------
        # APPEND TO CSV (NO MEMORY STORAGE)
        # -----------------------
        if not os.path.exists(OUTPUT_FILE):
            df.to_csv(OUTPUT_FILE, index=False)
        else:
            df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)

    except Exception:
        failed_games.append(game_id)

    time.sleep(0.8)

# -----------------------
# SAVE FAILURES
# -----------------------
pd.DataFrame(failed_games, columns=["GAME_ID"]).to_csv(FAILED_FILE, index=False)

print("Done streaming dataset")

NameError: name 'df_games' is not defined

In [ ]:
pip install tqdm

In [42]:
OUTPUT_FILE = "boxscores_stream.csv"
FAILED_FILE = "failed_games_retry.csv"

# -----------------------
# LOAD FAILED GAME IDS
# -----------------------
failed_ids = pd.read_csv("failed_games.csv")["GAME_ID"].tolist()

retry_failed = []

# -----------------------
# RETRY LOOP
# -----------------------
for game_id in tqdm(failed_ids, desc="Retrying failed games"):

    try:
        box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id)
        df = box.get_data_frames()[0]
        df["GAME_ID"] = game_id

        # append to existing dataset
        df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)

    except Exception:
        retry_failed.append(game_id)

    time.sleep(1.0)  # slightly slower for stability

# -----------------------
# SAVE STILL-FAILED
# -----------------------
pd.DataFrame(retry_failed, columns=["GAME_ID"]).to_csv(FAILED_FILE, index=False)

print(f"Recovered: {len(failed_ids) - len(retry_failed)}")
print(f"Still failed: {len(retry_failed)}")

Retrying failed games: 100%|██████████| 431/431 [09:48<00:00,  1.37s/it]

Recovered: 431
Still failed: 0


In [4]:
df = pd.read_csv("boxscores_stream.csv")

print(df.shape)
print(df.head())

(296771, 35)
     gameId      teamId teamCity teamName teamTricode teamSlug  personId  \
0  21500002  1610612741  Chicago    Bulls         CHI    bulls    203503   
1  21500002  1610612741  Chicago    Bulls         CHI    bulls    202703   
2  21500002  1610612741  Chicago    Bulls         CHI    bulls      2200   
3  21500002  1610612741  Chicago    Bulls         CHI    bulls    202710   
4  21500002  1610612741  Chicago    Bulls         CHI    bulls    201565   

  firstName  familyName          nameI  ... reboundsDefensive reboundsTotal  \
0      Tony       Snell       T. Snell  ...                 2             2   
1    Nikola     Mirotic     N. Mirotic  ...                 7             9   
2       Pau       Gasol       P. Gasol  ...                 2             2   
3     Jimmy  Butler III  J. Butler III  ...                 4             5   
4   Derrick        Rose        D. Rose  ...                 1             1   

  assists  steals blocks  turnovers  foulsPersonal  poi

In [6]:
cols = [
    "points",
    "fieldGoalsMade", "fieldGoalsAttempted",
    "threePointersMade", "threePointersAttempted",
    "freeThrowsMade", "freeThrowsAttempted",
    "reboundsTotal", "assists", "turnovers",
    "steals", "blocks"
]

df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

In [4]:
df = pd.read_csv("boxscores_stream.csv")

team_stats = (
    df.groupby(["GAME_ID","teamId", "teamTricode"], as_index=False)
    .agg({
        "points": "sum",
        "fieldGoalsMade": "sum",
        "fieldGoalsAttempted": "sum",
        "threePointersMade": "sum",
        "threePointersAttempted": "sum",
        "freeThrowsMade": "sum",
        "freeThrowsAttempted": "sum",
        "reboundsTotal": "sum",
        "assists": "sum",
        "turnovers": "sum",
        "steals": "sum",
        "blocks": "sum"
    })
)

In [6]:
print(team_stats.head())

    GAME_ID      teamId teamTricode  points  fieldGoalsMade  \
0  21500001  1610612737         ATL      94              37   
1  21500001  1610612765         DET     106              37   
2  21500002  1610612739         CLE      95              38   
3  21500002  1610612741         CHI      97              37   
4  21500003  1610612740         NOP      95              35   

   fieldGoalsAttempted  threePointersMade  threePointersAttempted  \
0                   82                  8                      27   
1                   96                 12                      29   
2                   94                  9                      29   
3                   87                  7                      19   
4                   83                  6                      18   

   freeThrowsMade  freeThrowsAttempted  reboundsTotal  assists  turnovers  \
0              12                   15             40       22         15   
1              20                   26             5

In [ ]:
team_stats.to_csv("team_stats.csv", index=False)

In [4]:
team_stats = pd.read_csv("team_stats.csv")
games = pd.read_csv("nba_games.csv")

# -----------------------
# MERGE HOME TEAM STATS
# -----------------------
home = pd.merge(
    games,
    team_stats,
    left_on=["GAME_ID", "TEAM_ABBREVIATION_HOME"],
    right_on=["GAME_ID", "teamTricode"],
    how="left"
)
# rename home columns
home = home.rename(columns={
    "points": "points_home",
    "reboundsTotal": "rebounds_home",
    "assists": "assists_home",
    "turnovers": "turnovers_home",
    "steals": "steals_home",
    "blocks": "blocks_home"
})

# -----------------------
# MERGE AWAY TEAM STATS
# -----------------------
full = pd.merge(
    home,
    team_stats,
    left_on=["GAME_ID", "TEAM_ABBREVIATION_AWAY"],
    right_on=["GAME_ID", "teamTricode"],
    how="left"
)

# rename away columns
full = full.rename(columns={
    "points": "points_away",
    "reboundsTotal": "rebounds_away",
    "assists": "assists_away",
    "turnovers": "turnovers_away",
    "steals": "steals_away",
    "blocks": "blocks_away"
})

# -----------------------
# CLEAN COLUMNS (optional)
# -----------------------
# drop duplicate / unnecessary columns
full = full.drop(columns=["teamTricode_x", "teamTricode_y"], errors="ignore")

# -----------------------
# SAVE FINAL DATASET
# -----------------------

print("Done. Final dataset shape:", full.shape)

Done. Final dataset shape: (11974, 32)


In [6]:
full["fg%_home"] = full["fieldGoalsMade_x"] / full["fieldGoalsAttempted_x"]
full["efg%_home"] = (full["fieldGoalsMade_x"] + (0.5 * 3*full["threePointersMade_x"])) / full["fieldGoalsAttempted_x"]

# AWAY efficiency
full["fg%_away"] = full["fieldGoalsMade_y"] / full["fieldGoalsAttempted_y"] 
full["efg%_away"] = (full["fieldGoalsMade_y"] + (0.5 * 3*full["threePointersMade_y"])) / full["fieldGoalsAttempted_y"]

In [ ]:
print(full.head())

In [8]:
full = full.drop(columns=["PTS_HOME", "PTS_AWAY"], errors = "ignore")

In [10]:
print(full.head())

    GAME_ID   GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
0  21500002  2015-10-27                    CHI                    CLE   
1  21500003  2015-10-27                    GSW                    NOP   
2  21500001  2015-10-27                    ATL                    DET   
3  21500005  2015-10-28                    BOS                    PHI   
4  21500010  2015-10-28                    HOU                    DEN   

       teamId_x  points_home  fieldGoalsMade_x  fieldGoalsAttempted_x  \
0  1.610613e+09         97.0              37.0                   87.0   
1  1.610613e+09        111.0              41.0                   96.0   
2  1.610613e+09         94.0              37.0                   82.0   
3  1.610613e+09        112.0              39.0                   85.0   
4  1.610613e+09         85.0              30.0                   87.0   

   threePointersMade_x  threePointersAttempted_x  ...  freeThrowsAttempted_y  \
0                  7.0                    

In [12]:
full["HOME_WIN"] = (full["points_home"] > full["points_away"]).astype(int)

In [173]:
full.to_csv("nba_model_dataset.csv", index=False)

NameError: name 'full' is not defined

In [ ]:
print(df.head())

In [62]:
import pandas as pd
df = pd.read_csv("nba_model_dataset.csv")


In [63]:
df = df.sort_values("GAME_DATE").reset_index(drop=True)

In [64]:
home = df[[
    "GAME_ID","GAME_DATE","TEAM_ABBREVIATION_HOME",
    "points_home","rebounds_home","assists_home",
    "turnovers_home","efg%_home"
]].copy()

home.columns = ["GAME_ID","GAME_DATE","TEAM_ID","PTS","REB","AST","TOV","EFG"]

away = df[[
    "GAME_ID","GAME_DATE","TEAM_ABBREVIATION_AWAY",
    "points_away","rebounds_away","assists_away",
    "turnovers_away","efg%_away"
]].copy()

away.columns = ["GAME_ID","GAME_DATE","TEAM_ID","PTS","REB","AST","TOV","EFG"]
home["IS_HOME"] = 1
away["IS_HOME"] = 0
long_df = pd.concat([home, away], ignore_index=True)

In [48]:
print(id(long_df))

4605933136


In [49]:


long_df["GAME_DATE"] = pd.to_datetime(long_df["GAME_DATE"])

long_df = long_df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

def roll(x):
    return x.shift(1).rolling(5).mean()

for col in ["PTS", "REB", "AST", "TOV", "EFG"]:
    long_df[f"{col}_LAST5"] = long_df.groupby("TEAM_ID")[col].transform(roll)

home_feat = long_df[long_df["IS_HOME"] == 1].rename(columns={
    "TEAM_ID":"TEAM_ABBREVIATION_HOME",
    "PTS_LAST5":"PTS_HOME_LAST5",
    "REB_LAST5":"REB_HOME_LAST5",
    "AST_LAST5":"AST_HOME_LAST5",
    "TOV_LAST5":"TOV_HOME_LAST5",
    "EFG_LAST5":"EFG_HOME_LAST5"
})

away_feat = long_df[long_df["IS_HOME"] == 0].rename(columns={
    "TEAM_ID":"TEAM_ABBREVIATION_AWAY",
    "PTS_LAST5":"PTS_AWAY_LAST5",
    "REB_LAST5":"REB_AWAY_LAST5",
    "AST_LAST5":"AST_AWAY_LAST5",
    "TOV_LAST5":"TOV_AWAY_LAST5",
    "EFG_LAST5":"EFG_AWAY_LAST5"
})

In [50]:
df_base = df[[
    "GAME_ID","GAME_DATE",
    "TEAM_ABBREVIATION_HOME",
    "TEAM_ABBREVIATION_AWAY",
    "HOME_WIN"
]].drop_duplicates("GAME_ID").copy()

In [51]:
check = df_base.merge(
    home_feat,
    on=["GAME_ID","TEAM_ABBREVIATION_HOME"],
    how="inner"
)

print("Matched rows:", check.shape[0])
print("Original rows:", df_base.shape[0])

Matched rows: 11974
Original rows: 11974


In [52]:
print(long_df[long_df["GAME_ID"] == df_base["GAME_ID"].iloc[0]])

       GAME_ID  GAME_DATE TEAM_ID   PTS   REB   AST   TOV       EFG  IS_HOME  \
3187  21500002 2015-10-27     CHI  97.0  47.0  13.0  13.0  0.545977        1   
3980  21500002 2015-10-27     CLE  95.0  50.0  26.0  10.0  0.547872        0   

      PTS_LAST5  REB_LAST5  AST_LAST5  TOV_LAST5  EFG_LAST5  
3187        NaN        NaN        NaN        NaN        NaN  
3980        NaN        NaN        NaN        NaN        NaN  


In [53]:
home_pts = long_df[long_df["IS_HOME"] == 1][["GAME_ID", "PTS"]] \
    .rename(columns={"PTS": "PTS_HOME"})

away_pts = long_df[long_df["IS_HOME"] == 0][["GAME_ID", "PTS"]] \
    .rename(columns={"PTS": "PTS_AWAY"})

df_final = df_base.merge(home_pts, on="GAME_ID", how="left") \
                   .merge(away_pts, on="GAME_ID", how="left")

df_final = df_base.merge(home_pts, on="GAME_ID", how="left") \
    .merge(away_pts, on="GAME_ID", how="left") \
    .merge(
        home_feat[
            ["GAME_ID","TEAM_ABBREVIATION_HOME",
             "PTS_HOME_LAST5","REB_HOME_LAST5",
             "AST_HOME_LAST5","TOV_HOME_LAST5","EFG_HOME_LAST5"]
        ],
        on=["GAME_ID","TEAM_ABBREVIATION_HOME"],
        how="left"
    ).merge(
        away_feat[
            ["GAME_ID","TEAM_ABBREVIATION_AWAY",
             "PTS_AWAY_LAST5","REB_AWAY_LAST5",
             "AST_AWAY_LAST5","TOV_AWAY_LAST5","EFG_AWAY_LAST5"]
        ],
        on=["GAME_ID","TEAM_ABBREVIATION_AWAY"],
        how="left"
    )

In [54]:
print(df_final.head(10))

    GAME_ID   GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
0  21500002  2015-10-27                    CHI                    CLE   
1  21500003  2015-10-27                    GSW                    NOP   
2  21500001  2015-10-27                    ATL                    DET   
3  21500011  2015-10-28                    MEM                    CLE   
4  21500013  2015-10-28                    OKC                    SAS   
5  21500016  2015-10-28                    SAC                    LAC   
6  21500014  2015-10-28                    PHX                    DAL   
7  21500008  2015-10-28                    MIA                    CHA   
8  21500007  2015-10-28                    DET                    UTA   
9  21500012  2015-10-28                    MIL                    NYK   

   HOME_WIN  PTS_HOME  PTS_AWAY  PTS_HOME_LAST5  REB_HOME_LAST5  \
0         1      97.0      95.0             NaN             NaN   
1         1     111.0      95.0             NaN             Na

In [55]:
df_final["PTS_DIFF"] = df_final["PTS_HOME_LAST5"] - df_final["PTS_AWAY_LAST5"]
df_final["REB_DIFF"] = df_final["REB_HOME_LAST5"] - df_final["REB_AWAY_LAST5"]
df_final["AST_DIFF"] = df_final["AST_HOME_LAST5"] - df_final["AST_AWAY_LAST5"]
df_final["TOV_DIFF"] = df_final["TOV_HOME_LAST5"] - df_final["TOV_AWAY_LAST5"]
df_final["EFG_DIFF"] = df_final["EFG_HOME_LAST5"] - df_final["EFG_AWAY_LAST5"]

In [56]:
print(df_final.head())

    GAME_ID   GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
0  21500002  2015-10-27                    CHI                    CLE   
1  21500003  2015-10-27                    GSW                    NOP   
2  21500001  2015-10-27                    ATL                    DET   
3  21500011  2015-10-28                    MEM                    CLE   
4  21500013  2015-10-28                    OKC                    SAS   

   HOME_WIN  PTS_HOME  PTS_AWAY  PTS_HOME_LAST5  REB_HOME_LAST5  \
0         1      97.0      95.0             NaN             NaN   
1         1     111.0      95.0             NaN             NaN   
2         0      94.0     106.0             NaN             NaN   
3         0      76.0     106.0             NaN             NaN   
4         1     112.0     106.0             NaN             NaN   

   AST_HOME_LAST5  ...  PTS_AWAY_LAST5  REB_AWAY_LAST5  AST_AWAY_LAST5  \
0             NaN  ...             NaN             NaN             NaN   
1         

In [78]:
import numpy as np
team_locations = {
    "ATL": (33.7490, -84.3880),
    "BOS": (42.3601, -71.0589),
    "BKN": (40.6782, -73.9442),
    "CHA": (35.2271, -80.8431),
    "CHI": (41.8781, -87.6298),
    "CLE": (41.4993, -81.6944),
    "DAL": (32.7767, -96.7970),
    "DEN": (39.7392, -104.9903),
    "DET": (42.3314, -83.0458),
    "GSW": (37.7749, -122.4194),
    "HOU": (29.7604, -95.3698),
    "IND": (39.7684, -86.1581),
    "LAC": (34.0522, -118.2437),
    "LAL": (34.0522, -118.2437),
    "MEM": (35.1495, -90.0490),
    "MIA": (25.7617, -80.1918),
    "MIL": (43.0389, -87.9065),
    "MIN": (44.9778, -93.2650),
    "NOP": (29.9511, -90.0715),
    "NYK": (40.7128, -74.0060),
    "OKC": (35.4676, -97.5164),
    "ORL": (28.5383, -81.3792),
    "PHI": (39.9526, -75.1652),
    "PHX": (33.4484, -112.0740),
    "POR": (45.5152, -122.6784),
    "SAC": (38.5816, -121.4944),
    "SAS": (29.4241, -98.4936),
    "TOR": (43.6532, -79.3832),
    "UTA": (40.7608, -111.8910),
    "WAS": (38.9072, -77.0369)
}

df_final["HOME_LAT"] = df_final["TEAM_ABBREVIATION_HOME"].map(lambda x: team_locations[x][0])
df_final["HOME_LON"] = df_final["TEAM_ABBREVIATION_HOME"].map(lambda x: team_locations[x][1])

df_final["AWAY_LAT"] = df_final["TEAM_ABBREVIATION_AWAY"].map(lambda x: team_locations[x][0])
df_final["AWAY_LON"] = df_final["TEAM_ABBREVIATION_AWAY"].map(lambda x: team_locations[x][1])

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

df_final["TRAVEL_DIST"] = haversine(
    df_final["AWAY_LAT"],
    df_final["AWAY_LON"],
    df_final["HOME_LAT"],
    df_final["HOME_LON"]
)

In [58]:
print(df_final.head())

    GAME_ID   GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
0  21500002  2015-10-27                    CHI                    CLE   
1  21500003  2015-10-27                    GSW                    NOP   
2  21500001  2015-10-27                    ATL                    DET   
3  21500011  2015-10-28                    MEM                    CLE   
4  21500013  2015-10-28                    OKC                    SAS   

   HOME_WIN  PTS_HOME  PTS_AWAY  PTS_HOME_LAST5  REB_HOME_LAST5  \
0         1      97.0      95.0             NaN             NaN   
1         1     111.0      95.0             NaN             NaN   
2         0      94.0     106.0             NaN             NaN   
3         0      76.0     106.0             NaN             NaN   
4         1     112.0     106.0             NaN             NaN   

   AST_HOME_LAST5  ...  PTS_DIFF  REB_DIFF  AST_DIFF  TOV_DIFF  EFG_DIFF  \
0             NaN  ...       NaN       NaN       NaN       NaN       NaN   
1     

In [59]:
features = ["PTS_HOME", "PTS_AWAY",
    "PTS_HOME_LAST5","REB_HOME_LAST5","AST_HOME_LAST5","TOV_HOME_LAST5","EFG_HOME_LAST5",
    "PTS_AWAY_LAST5","REB_AWAY_LAST5","AST_AWAY_LAST5","TOV_AWAY_LAST5","EFG_AWAY_LAST5"
]
df_final = df_final.dropna(subset=features)

In [60]:
print(df_final.head())

     GAME_ID   GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
71  21500074  2015-11-05                    POR                    MEM   
72  21500070  2015-11-05                    CHI                    OKC   
74  21500084  2015-11-06                    SAC                    HOU   
76  21500080  2015-11-06                    IND                    MIA   
77  21500083  2015-11-06                    GSW                    DEN   

    HOME_WIN  PTS_HOME  PTS_AWAY  PTS_HOME_LAST5  REB_HOME_LAST5  \
71         1     115.0      96.0           101.6            46.4   
72         1     104.0      98.0           100.6            43.2   
74         0     110.0     116.0           106.2            44.4   
76         1      90.0      87.0            94.4            41.4   
77         1     119.0     104.0           117.6            49.6   

    AST_HOME_LAST5  ...  PTS_DIFF  REB_DIFF  AST_DIFF  TOV_DIFF  EFG_DIFF  \
71            17.6  ...       9.4       2.8      -4.6       4.0  0.14

In [61]:
df_final.to_csv("nba_long_dataset.csv", index=False)

In [190]:
df_final = df_final.drop(columns=["HOME_ADV"], errors = "ignore")

In [70]:
long_df["GAME_DATE"] = pd.to_datetime(long_df["GAME_DATE"])

long_df = long_df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

def roll(x):
    return x.shift(1).rolling(50).mean()

for col in ["PTS", "REB", "AST", "TOV", "EFG"]:
    long_df[f"{col}_LAST50"] = long_df.groupby("TEAM_ID")[col].transform(roll)

home_feat = long_df[long_df["IS_HOME"] == 1].rename(columns={
    "TEAM_ID":"TEAM_ABBREVIATION_HOME",
    "PTS_LAST50":"PTS_HOME_LAST50",
    "REB_LAST50":"REB_HOME_LAST50",
    "AST_LAST50":"AST_HOME_LAST50",
    "TOV_LAST50":"TOV_HOME_LAST50",
    "EFG_LAST50":"EFG_HOME_LAST50"
})

away_feat = long_df[long_df["IS_HOME"] == 0].rename(columns={
    "TEAM_ID":"TEAM_ABBREVIATION_AWAY",
    "PTS_LAST50":"PTS_AWAY_LAST50",
    "REB_LAST50":"REB_AWAY_LAST50",
    "AST_LAST50":"AST_AWAY_LAST50",
    "TOV_LAST50":"TOV_AWAY_LAST50",
    "EFG_LAST50":"EFG_AWAY_LAST50"
})

In [71]:
print(home_feat.head(10))

     GAME_ID  GAME_DATE TEAM_ABBREVIATION_HOME    PTS   REB   AST   TOV  \
0   21500001 2015-10-27                    ATL   94.0  40.0  22.0  15.0   
2   21500026 2015-10-30                    ATL   97.0  45.0  23.0  15.0   
5   21500062 2015-11-04                    ATL  101.0  34.0  27.0  13.0   
7   21500086 2015-11-07                    ATL  114.0  39.0  37.0  14.0   
8   21500101 2015-11-09                    ATL  107.0  27.0  30.0  17.0   
9   21500117 2015-11-11                    ATL  106.0  44.0  26.0  12.0   
11  21500148 2015-11-15                    ATL   96.0  33.0  26.0  15.0   
13  21500169 2015-11-18                    ATL  103.0  45.0  23.0  11.0   
15  21500210 2015-11-24                    ATL  121.0  43.0  33.0  20.0   
19  21500256 2015-11-30                    ATL  106.0  51.0  23.0  12.0   

         EFG  IS_HOME  PTS_LAST20  REB_LAST20  AST_LAST20  TOV_LAST20  \
0   0.597561        1         NaN         NaN         NaN         NaN   
2   0.578313        1       

In [72]:
df_base = df[[
    "GAME_ID","GAME_DATE",
    "TEAM_ABBREVIATION_HOME",
    "TEAM_ABBREVIATION_AWAY",
    "HOME_WIN"
]].drop_duplicates("GAME_ID").copy()

In [73]:
check = df_base.merge(
    home_feat,
    on=["GAME_ID","TEAM_ABBREVIATION_HOME"],
    how="inner"
)

print("Matched rows:", check.shape[0])
print("Original rows:", df_base.shape[0])

Matched rows: 11974
Original rows: 11974


In [74]:
home_pts = long_df[long_df["IS_HOME"] == 1][["GAME_ID", "PTS"]] \
    .rename(columns={"PTS": "PTS_HOME"})

away_pts = long_df[long_df["IS_HOME"] == 0][["GAME_ID", "PTS"]] \
    .rename(columns={"PTS": "PTS_AWAY"})

df_final = df_base.merge(home_pts, on="GAME_ID", how="left") \
                   .merge(away_pts, on="GAME_ID", how="left")

df_final = df_base.merge(home_pts, on="GAME_ID", how="left") \
    .merge(away_pts, on="GAME_ID", how="left") \
    .merge(
        home_feat[
            ["GAME_ID","TEAM_ABBREVIATION_HOME",
             "PTS_HOME_LAST50","REB_HOME_LAST50",
             "AST_HOME_LAST50","TOV_HOME_LAST50","EFG_HOME_LAST50"]
        ],
        on=["GAME_ID","TEAM_ABBREVIATION_HOME"],
        how="left"
    ).merge(
        away_feat[
            ["GAME_ID","TEAM_ABBREVIATION_AWAY",
             "PTS_AWAY_LAST50","REB_AWAY_LAST50",
             "AST_AWAY_LAST50","TOV_AWAY_LAST50","EFG_AWAY_LAST50"]
        ],
        on=["GAME_ID","TEAM_ABBREVIATION_AWAY"],
        how="left"
    )

In [75]:
print(df_final.head(10))

    GAME_ID   GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
0  21500002  2015-10-27                    CHI                    CLE   
1  21500003  2015-10-27                    GSW                    NOP   
2  21500001  2015-10-27                    ATL                    DET   
3  21500011  2015-10-28                    MEM                    CLE   
4  21500013  2015-10-28                    OKC                    SAS   
5  21500016  2015-10-28                    SAC                    LAC   
6  21500014  2015-10-28                    PHX                    DAL   
7  21500008  2015-10-28                    MIA                    CHA   
8  21500007  2015-10-28                    DET                    UTA   
9  21500012  2015-10-28                    MIL                    NYK   

   HOME_WIN  PTS_HOME  PTS_AWAY  PTS_HOME_LAST50  REB_HOME_LAST50  \
0         1      97.0      95.0              NaN              NaN   
1         1     111.0      95.0              NaN          

In [80]:
features = ["PTS_HOME", "PTS_AWAY",
    "PTS_HOME_LAST50","REB_HOME_LAST50","AST_HOME_LAST50","TOV_HOME_LAST50","EFG_HOME_LAST50",
    "PTS_AWAY_LAST50","REB_AWAY_LAST50","AST_AWAY_LAST50","TOV_AWAY_LAST50","EFG_AWAY_LAST50"
]
df_final = df_final.dropna(subset=features)

In [81]:
df_final["PTS_DIFF"] = df_final["PTS_HOME_LAST50"] - df_final["PTS_AWAY_LAST50"]
df_final["REB_DIFF"] = df_final["REB_HOME_LAST50"] - df_final["REB_AWAY_LAST50"]
df_final["AST_DIFF"] = df_final["AST_HOME_LAST50"] - df_final["AST_AWAY_LAST50"]
df_final["TOV_DIFF"] = df_final["TOV_HOME_LAST50"] - df_final["TOV_AWAY_LAST50"]
df_final["EFG_DIFF"] = df_final["EFG_HOME_LAST50"] - df_final["EFG_AWAY_LAST50"]

In [82]:
df_final.to_csv("nba_long_dataset3.csv", index=False)